# Task 3: PCA Visualization of Uniform vs Non-uniform Embeddings

Одна интерактивная Plotly-визуализация для 31 временного ряда.

Сначала строятся полные embedding'и:

- non-uniform embedding по лагам из `pecora_unf_res.csv`;
- uniform embedding по `tau` и `dimension` из `acf_fnn_emb.csv`.

Затем каждый embedding отдельно проецируется в 3D через PCA, и уже PCA-проекция рисуется в Plotly. Dropdown синхронно переключает выбранный ряд на обоих графиках.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

pio.renderers.default = "plotly_mimetype+notebook"

DATA_PATH = Path("dataset_clean.csv")
UNIFORM_PATH = Path("uniform_dimensions.csv")
NONUNIFORM_PATH = Path("pecora_unf_res.csv")

MAX_POINTS = 3000
NORMALIZE_SERIES = True
MARKER_SIZE = 2.6
MARKER_OPACITY = 0.72

In [ ]:
# Данные временных рядов и результаты подбора параметров из предыдущего этапа.
# uniform_dimensions.csv хранит параметры равномерного вложения: tau и dimension.
# pecora_unf_res.csv хранит параметры неравномерного вложения: список лагов Pecora.
df = pd.read_csv(DATA_PATH)
uniform_params = pd.read_csv(UNIFORM_PATH)
nonuniform_params = pd.read_csv(NONUNIFORM_PATH)

if "TS" in df.columns:
    data_columns = [col for col in df.columns if col != "TS"]
else:
    data_columns = list(df.columns)

# Проверяем, что файлы с параметрами содержат все поля, нужные для восстановления embedding'ов.
uniform_required = {"column", "tau", "dimension"}
nonuniform_required = {"timeseries_name", "target_dim", "pecora_dim", "pecora_lags", "status"}

missing_uniform = uniform_required - set(uniform_params.columns)
missing_nonuniform = nonuniform_required - set(nonuniform_params.columns)

if missing_uniform:
    raise ValueError(f"Missing columns in {UNIFORM_PATH}: {sorted(missing_uniform)}")
if missing_nonuniform:
    raise ValueError(f"Missing columns in {NONUNIFORM_PATH}: {sorted(missing_nonuniform)}")

# Для сравнения берём только те ряды, для которых успешно найдены оба типа параметров.
nonuniform_ok = nonuniform_params.loc[nonuniform_params["status"].eq("ok")].copy()

common_columns = [
    col for col in uniform_params["column"].tolist()
    if col in data_columns and col in set(nonuniform_ok["timeseries_name"])
]

print(f"Loaded data: {df.shape[0]} rows")
print(f"Common series for visualization: {len(common_columns)}")

# 31 число рядов.
if len(common_columns) != 31:
    print("Warning: expected 31 common series")

In [ ]:
def prepare_series(df: pd.DataFrame, column: str, normalize: bool = True) -> np.ndarray:
    # Приводим ряд к числовому виду и убираем пропуски перед построением фазового пространства.
    series = pd.to_numeric(df[column]).astype(float)

    # Нормировка нужна, чтобы геометрия облака не зависела от масштаба конкретного датчика.
    if normalize:
        std = series.std()
        if std == 0:
            raise ValueError("constant series")
        series = (series - series.mean()) / std

    return series.to_numpy(dtype=float)


def delay_embedding(series: np.ndarray, m: int, tau: int) -> np.ndarray:
    # Равномерное вложение: каждая точка имеет вид
    # [x(t), x(t + tau), x(t + 2*tau), ..., x(t + (m-1)*tau)].
    if m < 3:
        raise ValueError("uniform embedding dimension must be >= 3 for 3D plot")
    if tau < 1:
        raise ValueError("tau must be >= 1")

    x = np.asarray(series, dtype=float)
    x = x[~np.isnan(x)]

    # Из-за лагов последние (m - 1) * tau наблюдений не могут стать началом полной точки.
    n_points = len(x) - (m - 1) * tau
    if n_points <= 0:
        raise ValueError(f"m={m}, tau={tau} are too large for series length {len(x)}")

    return np.column_stack([
        x[i * tau : i * tau + n_points]
        for i in range(m)
    ])


def nonuniform_embedding(series: np.ndarray, lags: list[int]) -> np.ndarray:
    # Неравномерное вложение: координаты берутся по разным лагам Pecora,
    # например [x(t), x(t + 14), x(t + 32), ...].
    if len(lags) < 3:
        raise ValueError("at least three lags are required for 3D plot")

    x = np.asarray(series, dtype=float)
    x = x[~np.isnan(x)]
    max_lag = max(lags)

    # Все координаты точки должны существовать, поэтому длину облака ограничивает max(lags).
    if len(x) <= max_lag:
        raise ValueError(f"lags={lags} are too large for series length {len(x)}")

    return np.column_stack([
        x[lag : len(x) - max_lag + lag]
        for lag in lags
    ])


def sample_for_plot(X: np.ndarray, max_points: int = MAX_POINTS) -> tuple[np.ndarray, np.ndarray]:
    # Подвыборка нужна только для интерактивности Plotly; сами embedding'и считаются полностью.
    if len(X) > max_points:
        idx = np.linspace(0, len(X) - 1, max_points).astype(int)
        return X[idx], idx
    return X, np.arange(len(X))


def pca_project(X: np.ndarray, n_components: int = 3) -> tuple[np.ndarray, np.ndarray]:
    # PCA используется только как метод 3D-проекции многомерного облака точек.
    # Параметры embedding'а при этом не переоцениваются.
    if X.shape[1] < n_components:
        raise ValueError(f"PCA needs at least {n_components} dimensions, got {X.shape[1]}")

    X_centered = X - X.mean(axis=0, keepdims=True)
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_centered)
    return X_pca, pca.explained_variance_ratio_


def parse_lags(value) -> list[int]:
    # В CSV лаги сохранены строкой вида "0;14;13;12".
    return [int(part) for part in str(value).split(";") if part != ""]

In [ ]:
# Индексируем таблицы параметров по имени ряда, чтобы для каждого датчика быстро достать
# tau/dimension для uniform embedding и список Pecora lags для non-uniform embedding.
uniform_by_column = uniform_params.set_index("column")
nonuniform_by_column = nonuniform_ok.set_index("timeseries_name")

embeddings = {}
errors = []

for column in common_columns:
    try:
        series = prepare_series(df, column, normalize=NORMALIZE_SERIES)

        # Uniform: используем оптимальные tau и dimension, найденные методом ACF + FNN.
        uniform_row = uniform_by_column.loc[column]
        uniform_tau = int(uniform_row["tau"])
        uniform_dim = int(uniform_row["dimension"])
        X_uniform = delay_embedding(series, m=uniform_dim, tau=uniform_tau)

        # Non-uniform: используем набор различных лагов, подобранный алгоритмом Pecora.
        # Размерность здесь равна числу лагов и определяется одновременно с ними.
        nonuniform_row = nonuniform_by_column.loc[column]
        lags = parse_lags(nonuniform_row["pecora_lags"])
        X_nonuniform = nonuniform_embedding(series, lags=lags)

        # Храним полные облака точек и метаданные, чтобы дальше сравнить их в одинаковой PCA-проекции.
        embeddings[column] = {
            "uniform": {
                "X": X_uniform,
                "tau": uniform_tau,
                "dimension": uniform_dim,
                "points": X_uniform.shape[0],
            },
            "nonuniform": {
                "X": X_nonuniform,
                "lags": lags,
                "dimension": int(nonuniform_row["pecora_dim"]),
                "points": X_nonuniform.shape[0],
            },
        }
    except Exception as exc:
        errors.append({"column": column, "error": str(exc)})

summary = pd.DataFrame([
    {
        "column": column,
        "uniform_tau": info["uniform"]["tau"],
        "uniform_dim": info["uniform"]["dimension"],
        "nonuniform_dim": info["nonuniform"]["dimension"],
        "nonuniform_lags": ";".join(map(str, info["nonuniform"]["lags"])),
        "uniform_points": info["uniform"]["points"],
        "nonuniform_points": info["nonuniform"]["points"],
    }
    for column, info in embeddings.items()
])

print(f"Built paired embeddings: {len(embeddings)}")
if errors:
    display(pd.DataFrame(errors))

display(summary)

In [ ]:
def format_var_ratio(ratio: np.ndarray) -> str:
    return ", ".join(f"PC{i + 1}={value:.1%}" for i, value in enumerate(ratio))


def make_comparison_figure(embeddings: dict[str, dict]) -> go.Figure:
    if not embeddings:
        raise ValueError("No paired embeddings to plot")

    columns = list(embeddings.keys())

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        horizontal_spacing=0.02,
        subplot_titles=("Non-uniform Pecora -> PCA", "Uniform ACF + FNN -> PCA"),
    )

    pca_meta = {}

    for index, column in enumerate(columns):
        info = embeddings[column]
        is_visible = index == 0

        # Проецируем каждое облако точек в PC1/PC2/PC3 отдельно.
        # Так сравниваем форму аттрактора после двух способов вложения.
        X_non_pca, non_ratio = pca_project(info["nonuniform"]["X"])
        X_non, t_non = sample_for_plot(X_non_pca)

        X_uni_pca, uni_ratio = pca_project(info["uniform"]["X"])
        X_uni, t_uni = sample_for_plot(X_uni_pca)

        pca_meta[column] = {
            "non_ratio": non_ratio,
            "uni_ratio": uni_ratio,
            "non_ratio_text": format_var_ratio(non_ratio),
            "uni_ratio_text": format_var_ratio(uni_ratio),
        }

        fig.add_trace(
            go.Scatter3d(
                x=X_non[:, 0],
                y=X_non[:, 1],
                z=X_non[:, 2],
                mode="markers",
                marker=dict(
                    size=MARKER_SIZE,
                    opacity=MARKER_OPACITY,
                    color=t_non,
                    colorscale="Turbo",
                    showscale=True,
                    colorbar=dict(title="t", x=0.47, len=0.72, thickness=14),
                ),
                name=f"non-uniform PCA: {column}",
                visible=is_visible,
                customdata=np.repeat(pca_meta[column]["non_ratio_text"], len(X_non)),
                hovertemplate=(
                    f"{column}<br>"
                    "PC1=%{x:.4g}<br>"
                    "PC2=%{y:.4g}<br>"
                    "PC3=%{z:.4g}<br>"
                    "t=%{marker.color}<br>"
                    "explained: %{customdata}<extra>non-uniform PCA</extra>"
                ),
                legendgroup=column,
                showlegend=False,
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter3d(
                x=X_uni[:, 0],
                y=X_uni[:, 1],
                z=X_uni[:, 2],
                mode="markers",
                marker=dict(
                    size=MARKER_SIZE,
                    opacity=MARKER_OPACITY,
                    color=t_uni,
                    colorscale="Turbo",
                    showscale=False,
                ),
                name=f"uniform PCA: {column}",
                visible=is_visible,
                customdata=np.repeat(pca_meta[column]["uni_ratio_text"], len(X_uni)),
                hovertemplate=(
                    f"{column}<br>"
                    "PC1=%{x:.4g}<br>"
                    "PC2=%{y:.4g}<br>"
                    "PC3=%{z:.4g}<br>"
                    "t=%{marker.color}<br>"
                    "explained: %{customdata}<extra>uniform PCA</extra>"
                ),
                legendgroup=column,
                showlegend=False,
            ),
            row=1,
            col=2,
        )

    buttons = []
    for index, column in enumerate(columns):
        info = embeddings[column]
        non = info["nonuniform"]
        uni = info["uniform"]

        visible = [False] * (2 * len(columns))
        visible[2 * index] = True
        visible[2 * index + 1] = True

        title = (
            f"{column}: non-uniform lags={non['lags']} | "
            f"uniform m={uni['dimension']}, tau={uni['tau']}"
        )

        buttons.append(
            dict(
                label=column,
                method="update",
                args=[
                    {"visible": visible},
                    {
                        "title": {"text": title, "x": 0.5},
                        "scene": dict(
                            xaxis_title="PC1",
                            yaxis_title="PC2",
                            zaxis_title="PC3",
                            aspectmode="data",
                        ),
                        "scene2": dict(
                            xaxis_title="PC1",
                            yaxis_title="PC2",
                            zaxis_title="PC3",
                            aspectmode="data",
                        ),
                    },
                ],
            )
        )

    first_column = columns[0]
    first = embeddings[first_column]

    fig.update_layout(
        title=dict(
            text=(
                f"{first_column}: non-uniform lags={first['nonuniform']['lags']} | "
                f"uniform m={first['uniform']['dimension']}, tau={first['uniform']['tau']}"
            ),
            x=0.5,
        ),
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                x=0.0,
                y=1.08,
                xanchor="left",
                yanchor="top",
                showactive=True,
                bgcolor="rgba(255,255,255,0.95)",
                bordercolor="rgba(70,70,70,0.25)",
                borderwidth=1,
            )
        ],
        scene=dict(
            xaxis_title="PC1",
            yaxis_title="PC2",
            zaxis_title="PC3",
            aspectmode="data",
            camera=dict(eye=dict(x=1.45, y=1.45, z=1.1)),
        ),
        scene2=dict(
            xaxis_title="PC1",
            yaxis_title="PC2",
            zaxis_title="PC3",
            aspectmode="data",
            camera=dict(eye=dict(x=1.45, y=1.45, z=1.1)),
        ),
        template="plotly_white",
        width=1250,
        height=720,
        margin=dict(l=10, r=10, b=10, t=105),
        font=dict(size=12),
    )

    fig.update_scenes(
        xaxis=dict(backgroundcolor="rgb(248,249,252)", gridcolor="rgb(225,229,235)", zerolinecolor="rgb(210,215,225)"),
        yaxis=dict(backgroundcolor="rgb(248,249,252)", gridcolor="rgb(225,229,235)", zerolinecolor="rgb(210,215,225)"),
        zaxis=dict(backgroundcolor="rgb(248,249,252)", gridcolor="rgb(225,229,235)", zerolinecolor="rgb(210,215,225)"),
    )

    return fig


comparison_fig = make_comparison_figure(embeddings)
comparison_fig.show()